# Ingestion 3: Object storage and manifests

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/ingestion/ingestion-03-object-storage.ipynb)

Publish a small, replayable file batch with a verifiable manifest.

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q boto3 moto[s3]

## Setup and sample data

Two small CSV objects form one delivery. Moto supplies an in-process S3 service.

In [ ]:
import hashlib
import json

import boto3
from moto import mock_aws

objects = {"batch-42/part-1.csv": b"id\n1\n", "batch-42/part-2.csv": b"id\n2\n"}

## Publish a manifest after the files

Write immutable payload keys, then publish a manifest listing the exact objects and checksums. A consumer uses the manifest as its read boundary.

**Where it matters**

Tell consumers which files belong to a complete delivery.

**Watch out for**

A prefix listing can expose only some files while the producer is still uploading. Independently verify checksums and retain manifests for replay.

In [ ]:
with mock_aws():
    s3 = boto3.client("s3", region_name="us-east-1")
    bucket = "handbook-landing"
    s3.create_bucket(Bucket=bucket)
    entries = []
    for key, payload in objects.items():
        s3.put_object(Bucket=bucket, Key=key, Body=payload)
        entries.append({"key": key, "sha256": hashlib.sha256(payload).hexdigest()})
    s3.put_object(Bucket=bucket, Key="batch-42/manifest.json", Body=json.dumps(entries).encode())
    manifest = json.loads(s3.get_object(Bucket=bucket, Key="batch-42/manifest.json")["Body"].read())
    for entry in manifest:
        payload = s3.get_object(Bucket=bucket, Key=entry["key"])["Body"].read()
        print(entry["key"], hashlib.sha256(payload).hexdigest() == entry["sha256"])

**What to notice:** Both listed data objects exist and their hashes match the manifest.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

A worker sees a data object but no manifest. Is the batch ready?

In [ ]:
# Try your solution here.

In [ ]:
print("No. Wait for the manifest before reading the delivery.")

## Key takeaways

- Tell consumers which files belong to a complete delivery.

**Next:** [Ingestion 4: dlt](ingestion-04-dlt.ipynb)